In [1]:
# ============================================================
# KAIST 대학원 RAG 챗봇 성능 평가 종합 코드
# 실행 위치: notebooks 폴더 안의 .ipynb 권장
# 결과 저장: notebooks/chatbot_eval_outputs/chatbot_test_results.csv
# ============================================================

from pathlib import Path
import sys
import time
import json
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# 1. 프로젝트 경로 설정
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)


# ============================================================
# 2. 기존 RagPipeline 불러오기
# ============================================================

from src.rag.rag_pipeline import create_default_pipeline

pipeline = create_default_pipeline(
    include_sql=True,
    include_debug_context=True,
    preload_vector_retriever=True,
)

print("RagPipeline 로드 완료")
print("Pipeline type:", type(pipeline))

print("RagPipeline 로드 완료")
print("Pipeline type:", type(pipeline))

available_methods = [
    name for name in dir(pipeline)
    if not name.startswith("_")
]

print("사용 가능한 메서드:")
print(available_methods)


# ============================================================
# 3. 테스트 질문 100개
# ============================================================

test_questions = [
    # ========================================================
    # 1. AI대학 / 학과 기본 소개
    # ========================================================
    {"id": 1, "category": "AI대학 소개", "question": "KAIST AI대학은 어떤 곳이야?", "expected": "answer"},
    {"id": 2, "category": "AI대학 소개", "question": "AI컴퓨팅학과는 어떤 학과야?", "expected": "answer"},
    {"id": 3, "category": "AI대학 소개", "question": "AI시스템학과는 어떤 분야를 다루는 학과야?", "expected": "answer"},
    {"id": 4, "category": "AI대학 소개", "question": "AX학과는 어떤 목적을 가진 학과야?", "expected": "answer"},
    {"id": 5, "category": "AI대학 소개", "question": "AI미래학과는 어떤 인재를 양성하려고 해?", "expected": "answer"},
    {"id": 6, "category": "AI대학 소개", "question": "AI컴퓨팅학과의 주요 특징을 설명해줘.", "expected": "answer"},
    {"id": 7, "category": "AI대학 소개", "question": "AI시스템학과의 주요 특징을 설명해줘.", "expected": "answer"},
    {"id": 8, "category": "AI대학 소개", "question": "AX학과의 주요 특징을 설명해줘.", "expected": "answer"},
    {"id": 9, "category": "AI대학 소개", "question": "AI미래학과의 주요 특징을 설명해줘.", "expected": "answer"},
    {"id": 10, "category": "AI대학 소개", "question": "AI대학에 속한 학과들을 정리해줘.", "expected": "answer"},

    # ========================================================
    # 2. 학과 비교
    # ========================================================
    {"id": 11, "category": "학과 비교", "question": "AI컴퓨팅학과와 AI시스템학과의 차이를 설명해줘.", "expected": "answer"},
    {"id": 12, "category": "학과 비교", "question": "AX학과와 AI미래학과는 어떤 점이 달라?", "expected": "answer"},
    {"id": 13, "category": "학과 비교", "question": "AI컴퓨팅학과와 AX학과를 비교해줘.", "expected": "answer"},
    {"id": 14, "category": "학과 비교", "question": "AI시스템학과와 AI미래학과를 비교해줘.", "expected": "answer"},
    {"id": 15, "category": "학과 비교", "question": "AI대학 학과별 교육 방향을 비교해서 알려줘.", "expected": "answer"},
    {"id": 16, "category": "학과 비교", "question": "AI대학 학과별 핵심 키워드를 정리해줘.", "expected": "answer"},
    {"id": 17, "category": "학과 비교", "question": "AI컴퓨팅학과는 다른 AI대학 학과들과 무엇이 달라?", "expected": "answer"},
    {"id": 18, "category": "학과 비교", "question": "AI시스템학과는 다른 AI대학 학과들과 무엇이 달라?", "expected": "answer"},
    {"id": 19, "category": "학과 비교", "question": "AI대학 학과들을 표 형식으로 비교해줘.", "expected": "answer"},
    {"id": 20, "category": "학과 비교", "question": "AI대학 학과별로 어떤 학생에게 적합한지 알려줘.", "expected": "answer"},

    # ========================================================
    # 3. 입학 정보
    # ========================================================
    {"id": 21, "category": "입학 정보", "question": "AI컴퓨팅학과 입학 정보를 알려줘.", "expected": "answer"},
    {"id": 22, "category": "입학 정보", "question": "AI시스템학과 입학 정보를 알려줘.", "expected": "answer"},
    {"id": 23, "category": "입학 정보", "question": "AX학과 입학 정보를 알려줘.", "expected": "answer"},
    {"id": 24, "category": "입학 정보", "question": "AI미래학과 입학 정보를 알려줘.", "expected": "answer"},
    {"id": 25, "category": "입학 정보", "question": "AI대학 학과별 입학 정보를 비교해줘.", "expected": "answer"},
    {"id": 26, "category": "입학 정보", "question": "AI대학 입학 지원 절차가 문서에 나와 있으면 알려줘.", "expected": "answer"},
    {"id": 27, "category": "입학 정보", "question": "AI대학 입학 지원 자격이 문서에 나와 있어?", "expected": "answer"},
    {"id": 28, "category": "입학 정보", "question": "AI대학 입학 지원 일정이 문서에 나와 있으면 알려줘.", "expected": "answer"},
    {"id": 29, "category": "입학 정보", "question": "AI대학 입학 관련 문의처가 있으면 알려줘.", "expected": "answer"},
    {"id": 30, "category": "입학 정보", "question": "AI대학 석사와 박사 입학 정보가 구분되어 있는지 알려줘.", "expected": "answer"},

    # ========================================================
    # 4. 교육과정 / 교과목
    # ========================================================
    {"id": 31, "category": "교육과정", "question": "AI컴퓨팅학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 32, "category": "교육과정", "question": "AI시스템학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 33, "category": "교육과정", "question": "AX학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 34, "category": "교육과정", "question": "AI미래학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 35, "category": "교육과정", "question": "AI대학에서 제공하는 교과목 정보를 정리해줘.", "expected": "answer"},
    {"id": 36, "category": "교육과정", "question": "AI컴퓨팅학과에서 배울 수 있는 과목을 알려줘.", "expected": "answer"},
    {"id": 37, "category": "교육과정", "question": "AI시스템학과에서 배울 수 있는 과목을 알려줘.", "expected": "answer"},
    {"id": 38, "category": "교육과정", "question": "AX학과에서 배울 수 있는 과목을 알려줘.", "expected": "answer"},
    {"id": 39, "category": "교육과정", "question": "AI미래학과에서 배울 수 있는 과목을 알려줘.", "expected": "answer"},
    {"id": 40, "category": "교육과정", "question": "AI대학 학과별 주요 교과목을 비교해줘.", "expected": "answer"},

    # ========================================================
    # 5. 연구 분야 / 교수진
    # ========================================================
    {"id": 41, "category": "교수진/연구", "question": "AI컴퓨팅학과 교수진 정보를 알려줘.", "expected": "answer"},
    {"id": 42, "category": "교수진/연구", "question": "AI시스템학과 교수진 정보를 알려줘.", "expected": "answer"},
    {"id": 43, "category": "교수진/연구", "question": "AX학과 교수진 정보를 알려줘.", "expected": "answer"},
    {"id": 44, "category": "교수진/연구", "question": "AI미래학과 교수진 정보를 알려줘.", "expected": "answer"},
    {"id": 45, "category": "교수진/연구", "question": "AI대학 교수진 목록을 정리해줘.", "expected": "answer"},
    {"id": 46, "category": "교수진/연구", "question": "AI대학 교수들의 연구 분야를 알려줘.", "expected": "answer"},
    {"id": 47, "category": "교수진/연구", "question": "AI컴퓨팅 관련 연구를 하는 교수는 누구야?", "expected": "answer"},
    {"id": 48, "category": "교수진/연구", "question": "AI시스템 관련 연구를 하는 교수는 누구야?", "expected": "answer"},
    {"id": 49, "category": "교수진/연구", "question": "AX 관련 연구를 하는 교수진 정보가 있으면 알려줘.", "expected": "answer"},
    {"id": 50, "category": "교수진/연구", "question": "AI대학 교수 홈페이지나 이메일 정보가 있으면 알려줘.", "expected": "answer"},

    # ========================================================
    # 6. 졸업 / 이수 요건
    # ========================================================
    {"id": 51, "category": "졸업/이수", "question": "AI컴퓨팅학과의 졸업 요건이 문서에 나와 있어?", "expected": "answer"},
    {"id": 52, "category": "졸업/이수", "question": "AI시스템학과의 졸업 요건을 알려줘.", "expected": "answer"},
    {"id": 53, "category": "졸업/이수", "question": "AX학과의 졸업 요건을 알려줘.", "expected": "answer"},
    {"id": 54, "category": "졸업/이수", "question": "AI미래학과의 졸업 요건을 알려줘.", "expected": "answer"},
    {"id": 55, "category": "졸업/이수", "question": "AI대학 학과별 이수 요건을 비교해줘.", "expected": "answer"},
    {"id": 56, "category": "졸업/이수", "question": "AI대학 석사 과정 이수 요건이 있으면 알려줘.", "expected": "answer"},
    {"id": 57, "category": "졸업/이수", "question": "AI대학 박사 과정 이수 요건이 있으면 알려줘.", "expected": "answer"},
    {"id": 58, "category": "졸업/이수", "question": "AI대학 필수 과목 정보가 문서에 나와 있어?", "expected": "answer"},
    {"id": 59, "category": "졸업/이수", "question": "AI대학 논문 제출 요건이 문서에 나와 있으면 알려줘.", "expected": "answer"},
    {"id": 60, "category": "졸업/이수", "question": "AI대학 수료 요건과 졸업 요건을 구분해서 설명해줘.", "expected": "answer"},

    # ========================================================
    # 7. 추천형 질문
    # ========================================================
    {"id": 61, "category": "추천", "question": "나는 AI 엔지니어가 되고 싶은데 AI대학에서 어떤 학과가 적합할까?", "expected": "answer"},
    {"id": 62, "category": "추천", "question": "나는 대규모 AI 시스템 개발에 관심이 있는데 어떤 학과가 좋아?", "expected": "answer"},
    {"id": 63, "category": "추천", "question": "나는 AI 모델 연구보다 AI 서비스 적용에 관심이 있는데 어떤 학과가 맞아?", "expected": "answer"},
    {"id": 64, "category": "추천", "question": "나는 산업 현장에 AI를 적용하는 분야에 관심이 있어. 어떤 학과가 적합해?", "expected": "answer"},
    {"id": 65, "category": "추천", "question": "나는 AI 반도체나 AI 하드웨어보다 소프트웨어 쪽에 관심이 있어. 어떤 학과를 보면 좋을까?", "expected": "answer"},
    {"id": 66, "category": "추천", "question": "나는 AI 시스템 구조와 인프라에 관심이 있어. 어떤 학과가 적합해?", "expected": "answer"},
    {"id": 67, "category": "추천", "question": "나는 미래 AI 기술과 사회 변화에 관심이 있는데 어떤 학과가 맞아?", "expected": "answer"},
    {"id": 68, "category": "추천", "question": "AI대학 학과 중 실무 적용 중심으로 보이는 학과를 추천해줘.", "expected": "answer"},
    {"id": 69, "category": "추천", "question": "AI대학 학과 중 연구 중심으로 보이는 학과를 추천해줘.", "expected": "answer"},
    {"id": 70, "category": "추천", "question": "내가 AI 서비스 기획과 개발을 같이 하고 싶다면 어떤 학과 정보를 먼저 봐야 해?", "expected": "answer"},

    # ========================================================
    # 8. 행정 / 위치 / 연락처
    # ========================================================
    {"id": 71, "category": "행정정보", "question": "AI대학의 위치 정보가 있으면 알려줘.", "expected": "answer"},
    {"id": 72, "category": "행정정보", "question": "AI대학의 연락처 정보를 알려줘.", "expected": "answer"},
    {"id": 73, "category": "행정정보", "question": "AI컴퓨팅학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 74, "category": "행정정보", "question": "AI시스템학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 75, "category": "행정정보", "question": "AX학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 76, "category": "행정정보", "question": "AI미래학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 77, "category": "행정정보", "question": "AI대학 홈페이지 주소를 알려줘.", "expected": "answer"},
    {"id": 78, "category": "행정정보", "question": "AI대학 입학 문의 이메일이나 전화번호가 있으면 알려줘.", "expected": "answer"},
    {"id": 79, "category": "행정정보", "question": "AI대학 행정실 정보가 문서에 나와 있으면 알려줘.", "expected": "answer"},
    {"id": 80, "category": "행정정보", "question": "AI대학 학과별 홈페이지 URL을 정리해줘.", "expected": "answer"},

    # ========================================================
    # 9. 출처 / 근거 기반 답변
    # ========================================================
    {"id": 81, "category": "출처 확인", "question": "AI컴퓨팅학과 소개를 출처와 함께 설명해줘.", "expected": "answer"},
    {"id": 82, "category": "출처 확인", "question": "AI시스템학과 소개를 출처와 함께 설명해줘.", "expected": "answer"},
    {"id": 83, "category": "출처 확인", "question": "AX학과 소개를 출처와 함께 설명해줘.", "expected": "answer"},
    {"id": 84, "category": "출처 확인", "question": "AI미래학과 소개를 출처와 함께 설명해줘.", "expected": "answer"},
    {"id": 85, "category": "출처 확인", "question": "AI대학 학과별 교육과정 정보를 출처와 함께 알려줘.", "expected": "answer"},
    {"id": 86, "category": "출처 확인", "question": "AI대학 교수진 정보를 출처와 함께 알려줘.", "expected": "answer"},
    {"id": 87, "category": "출처 확인", "question": "AI대학 입학 정보를 문서 근거와 함께 설명해줘.", "expected": "answer"},
    {"id": 88, "category": "출처 확인", "question": "AI대학 졸업 요건을 문서 근거와 함께 설명해줘.", "expected": "answer"},
    {"id": 89, "category": "출처 확인", "question": "AI대학 학과 비교 내용을 출처 기반으로 설명해줘.", "expected": "answer"},
    {"id": 90, "category": "출처 확인", "question": "AI대학 관련 답변은 문서에서 확인되는 내용만 사용해서 설명해줘.", "expected": "answer"},

    # ========================================================
    # 10. 환각 방지 / 문서 외 질문
    # ========================================================
    {"id": 91, "category": "환각 방지", "question": "AI대학의 올해 경쟁률을 알려줘.", "expected": "reject"},
    {"id": 92, "category": "환각 방지", "question": "AI대학 등록금이 얼마인지 알려줘.", "expected": "reject"},
    {"id": 93, "category": "환각 방지", "question": "AI대학 졸업생 평균 연봉을 알려줘.", "expected": "reject"},
    {"id": 94, "category": "환각 방지", "question": "AI대학 취업률을 알려줘.", "expected": "reject"},
    {"id": 95, "category": "환각 방지", "question": "AI대학 교수님별 논문 실적 순위를 알려줘.", "expected": "reject"},
    {"id": 96, "category": "환각 방지", "question": "내 학점이 3.2인데 AI대학에 합격할 수 있을까?", "expected": "reject"},
    {"id": 97, "category": "환각 방지", "question": "AI대학에서 가장 합격하기 쉬운 학과는 어디야?", "expected": "reject"},
    {"id": 98, "category": "환각 방지", "question": "AI대학 장학금 지급 금액을 정확히 알려줘.", "expected": "reject"},
    {"id": 99, "category": "환각 방지", "question": "서울대학교 AI대학원과 KAIST AI대학을 교수진 기준으로 비교해줘.", "expected": "reject"},
    {"id": 100, "category": "환각 방지", "question": "AI대학 졸업 후 대기업 취업 가능성을 퍼센트로 알려줘.", "expected": "reject"},
]

test_df = pd.DataFrame(test_questions)

print("테스트 질문 개수:", len(test_df))
display(test_df.head())
display(test_df.tail())


# ============================================================
# 4. Pipeline 호출 함수
# ============================================================

def call_pipeline(question):
    if hasattr(pipeline, "run_dict"):
        return pipeline.run_dict(
            question,
            include_debug_context=True,
        )

    if hasattr(pipeline, "run"):
        return pipeline.run(question)

    raise AttributeError("pipeline에 run_dict 또는 run 메서드가 없습니다.")


# ============================================================
# 5. 답변 / 출처 추출 함수
# ============================================================

def safe_to_string(value):
    """
    CSV 저장을 위해 값을 안전하게 문자열로 변환한다.
    """
    if value is None:
        return ""

    if isinstance(value, str):
        return value

    try:
        return json.dumps(value, ensure_ascii=False, default=str)
    except Exception:
        return str(value)


def extract_answer(response):
    """
    pipeline 응답에서 답변 텍스트를 추출한다.
    """

    if isinstance(response, str):
        return response

    if isinstance(response, dict):
        answer_keys = [
            "answer",
            "final_answer",
            "response",
            "result",
            "output",
            "content",
            "message"
        ]

        for key in answer_keys:
            if key in response:
                return safe_to_string(response[key])

        return safe_to_string(response)

    if hasattr(response, "content"):
        return safe_to_string(response.content)

    return safe_to_string(response)


def normalize_doc_list(value):
    """
    response 안의 문서 후보들을 list 형태로 정리한다.
    """
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    return [value]


def extract_source_from_doc(doc):
    """
    Document 객체 또는 dict에서 출처 문자열을 추출한다.
    """

    # LangChain Document 형태
    if hasattr(doc, "metadata"):
        metadata = doc.metadata or {}

        source = metadata.get("source", "unknown")
        page = metadata.get("page", "")
        title = metadata.get("title", "")
        school = metadata.get("school", "")
        category = metadata.get("category", "")
        url = metadata.get("url", "")

        parts = [f"source={source}"]

        if page != "":
            parts.append(f"page={page}")
        if title:
            parts.append(f"title={title}")
        if school:
            parts.append(f"school={school}")
        if category:
            parts.append(f"category={category}")
        if url:
            parts.append(f"url={url}")

        return " | ".join(parts)

    # dict 형태
    if isinstance(doc, dict):
        metadata = doc.get("metadata", {})

        if isinstance(metadata, dict):
            source = (
                metadata.get("source")
                or metadata.get("url")
                or metadata.get("file")
                or metadata.get("title")
                or doc.get("source")
                or doc.get("url")
                or doc.get("file")
                or doc.get("title")
                or ""
            )
        else:
            source = (
                doc.get("source")
                or doc.get("url")
                or doc.get("file")
                or doc.get("title")
                or ""
            )

        if source:
            return safe_to_string(source)

        return safe_to_string(doc)[:300]

    return safe_to_string(doc)[:300]


def extract_sources(response):
    """
    pipeline 응답에서 출처 정보를 추출한다.
    """

    sources = []

    if not isinstance(response, dict):
        return sources

    # 1. sources key가 직접 있는 경우
    if "sources" in response:
        raw_sources = response["sources"]

        if isinstance(raw_sources, list):
            for src in raw_sources:
                sources.append(safe_to_string(src))
        elif raw_sources:
            sources.append(safe_to_string(raw_sources))

    # 2. 검색 문서가 들어 있을 가능성이 있는 key들
    possible_doc_keys = [
        "source_documents",
        "documents",
        "docs",
        "context",
        "contexts",
        "retrieved_docs",
        "retrieved_documents",
        "search_results",
        "vector_results",
        "results"
    ]

    docs = []

    for key in possible_doc_keys:
        if key in response:
            docs.extend(normalize_doc_list(response[key]))

    for doc in docs:
        sources.append(extract_source_from_doc(doc))

    # 3. 중복 제거
    sources = [src for src in sources if src]
    sources = list(dict.fromkeys(sources))

    return sources


# ============================================================
# 6. 단일 질문 테스트
# ============================================================

test_question = "인공지능반도체대학원의 교육 목표를 설명해줘."

print("\n단일 질문 테스트 시작")
print("질문:", test_question)

try:
    raw_response = call_pipeline(test_question)

    answer = extract_answer(raw_response)
    sources = extract_sources(raw_response)

    print("\n[답변]")
    print(answer)

    print("\n[출처]")
    if sources:
        for source in sources:
            print("-", source)
    else:
        print("출처 추출 결과 없음")

    print("\n[raw_response 타입]")
    print(type(raw_response))

    if isinstance(raw_response, dict):
        print("\n[raw_response keys]")
        print(list(raw_response.keys()))

except Exception as e:
    print("단일 질문 테스트 실패")
    raise e


# ============================================================
# 7. 100개 질문 일괄 평가
# ============================================================

results = []

print("\n100개 질문 평가 시작")

for row in tqdm(test_questions):
    question_id = row["id"]
    category = row["category"]
    question = row["question"]
    expected = row["expected"]

    start_time = time.time()

    try:
        raw_response = call_pipeline(question)
        elapsed_time = round(time.time() - start_time, 3)

        answer = extract_answer(raw_response)
        sources = extract_sources(raw_response)

        results.append({
            "id": question_id,
            "category": category,
            "question": question,
            "expected": expected,
            "answer": answer,
            "sources": " | ".join(sources),
            "source_count": len(sources),
            "latency_sec": elapsed_time,
            "error": ""
        })

    except Exception as e:
        elapsed_time = round(time.time() - start_time, 3)

        results.append({
            "id": question_id,
            "category": category,
            "question": question,
            "expected": expected,
            "answer": "",
            "sources": "",
            "source_count": 0,
            "latency_sec": elapsed_time,
            "error": str(e)
        })

result_df = pd.DataFrame(results)

print("평가 완료:", len(result_df), "개")
display(result_df.head())


# ============================================================
# 8. 자동 평가 컬럼 추가
# ============================================================

refusal_keywords = [
    "확인되지",
    "찾을 수 없",
    "제공된 문서",
    "문서에 없",
    "자료에 없",
    "근거가 없",
    "알 수 없",
    "확인이 필요",
    "포함되어 있지",
    "제공된 자료",
    "답변할 수 없",
    "제공된 정보",
    "확인할 수 없",
    "확인하기 어렵",
    "명시되어 있지",
    "나와 있지",
]


def contains_refusal(answer):
    answer = str(answer)
    return any(keyword in answer for keyword in refusal_keywords)


def basic_evaluate(row):
    answer = str(row["answer"])
    expected = row["expected"]
    error = str(row["error"])

    if error:
        return "ERROR"

    if not answer.strip():
        return "EMPTY"

    if expected == "reject":
        if contains_refusal(answer):
            return "PASS_REJECT"
        else:
            return "FAIL_POSSIBLE_HALLUCINATION"

    if expected == "answer":
        if len(answer.strip()) < 20:
            return "WEAK_ANSWER"
        else:
            return "PASS_ANSWER"

    return "UNKNOWN"


result_df["auto_eval"] = result_df.apply(basic_evaluate, axis=1)

print("\n자동 평가 결과")
display(result_df[["id", "category", "question", "expected", "auto_eval", "latency_sec"]].head())


# ============================================================
# 9. 요약 통계
# ============================================================

overall_summary = (
    result_df["auto_eval"]
    .value_counts()
    .reset_index()
)

overall_summary.columns = ["auto_eval", "count"]

category_summary = (
    result_df
    .groupby(["category", "auto_eval"])
    .size()
    .reset_index(name="count")
)

print("\n전체 평가 요약")
display(overall_summary)

print("\n카테고리별 평가 요약")
display(category_summary)

print("\n기본 지표")
print("전체 질문 수:", len(result_df))
print("평균 응답 시간:", round(result_df["latency_sec"].mean(), 3), "초")
print("최대 응답 시간:", round(result_df["latency_sec"].max(), 3), "초")
print("에러 개수:", (result_df["auto_eval"] == "ERROR").sum())
print("빈 답변 개수:", (result_df["auto_eval"] == "EMPTY").sum())
print("환각 의심 개수:", (result_df["auto_eval"] == "FAIL_POSSIBLE_HALLUCINATION").sum())


# ============================================================
# 10. 문제 있는 답변만 확인
# ============================================================

problem_df = result_df[
    result_df["auto_eval"].isin([
        "ERROR",
        "EMPTY",
        "WEAK_ANSWER",
        "FAIL_POSSIBLE_HALLUCINATION"
    ])
]

print("\n문제 의심 답변 개수:", len(problem_df))

if len(problem_df) > 0:
    display(problem_df[[
        "id",
        "category",
        "question",
        "expected",
        "answer",
        "sources",
        "auto_eval",
        "error"
    ]])
else:
    print("문제 의심 답변이 없습니다.")


# ============================================================
# 11. 특정 질문 상세 확인 함수
# ============================================================

def show_answer(question_id):
    matched = result_df[result_df["id"] == question_id]

    if matched.empty:
        print(f"{question_id}번 질문을 찾을 수 없습니다.")
        return

    row = matched.iloc[0]

    print("=" * 100)
    print(f"[ID] {row['id']}")
    print(f"[CATEGORY] {row['category']}")
    print(f"[EXPECTED] {row['expected']}")
    print(f"[AUTO_EVAL] {row['auto_eval']}")
    print(f"[LATENCY] {row['latency_sec']} sec")
    print("-" * 100)

    print("[QUESTION]")
    print(row["question"])
    print("-" * 100)

    print("[ANSWER]")
    print(row["answer"])
    print("-" * 100)

    print("[SOURCES]")
    print(row["sources"])
    print("-" * 100)

    print("[ERROR]")
    print(row["error"])
    print("=" * 100)


# 예시 확인
print("\n예시 답변 확인: 1번, 91번")
show_answer(1)
show_answer(91)


# ============================================================
# 12. CSV 저장
# ============================================================

if CURRENT_DIR.name == "notebooks":
    NOTEBOOKS_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOKS_DIR = CURRENT_DIR / "notebooks"
else:
    NOTEBOOKS_DIR = CURRENT_DIR

OUT_DIR = NOTEBOOKS_DIR / "chatbot_eval_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUT_DIR / "chatbot_test_results.csv"
summary_path = OUT_DIR / "chatbot_test_summary.csv"
problem_path = OUT_DIR / "chatbot_test_problems.csv"

result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
overall_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
problem_df.to_csv(problem_path, index=False, encoding="utf-8-sig")

print("\nCSV 저장 완료")
print("전체 결과:", csv_path)
print("요약 결과:", summary_path)
print("문제 답변:", problem_path)

CURRENT_DIR: C:\Users\Playdata\workspace\SKN28-third-2TEAM\notebooks
PROJECT_ROOT: C:\Users\Playdata\workspace\SKN28-third-2TEAM
RagPipeline 로드 완료
Pipeline type: <class 'src.rag.rag_pipeline.RagPipeline'>
RagPipeline 로드 완료
Pipeline type: <class 'src.rag.rag_pipeline.RagPipeline'>
사용 가능한 메서드:
['analyzer', 'answer_config', 'build_context', 'classify_question', 'config', 'context_config', 'generate_answer', 'generate_answer_streaming', 'last_warm_up_result', 'run', 'run_dict', 'run_streaming', 'search', 'sql_retriever', 'vector_config', 'warm_up']
테스트 질문 개수: 100


,id,category,question,expected
0,1,AI대학 소개,KAIST AI대학은 어떤 곳이야?,answer
1,2,AI대학 소개,AI컴퓨팅학과는 어떤 학과야?,answer
2,3,AI대학 소개,AI시스템학과는 어떤 분야를 다루는 학과야?,answer
3,4,AI대학 소개,AX학과는 어떤 목적을 가진 학과야?,answer
4,5,AI대학 소개,AI미래학과는 어떤 인재를 양성하려고 해?,answer


,id,category,question,expected
95,96,환각 방지,내 학점이 3.2인데 AI대학에 합격할 수 있을까?,reject
96,97,환각 방지,AI대학에서 가장 합격하기 쉬운 학과는 어디야?,reject
97,98,환각 방지,AI대학 장학금 지급 금액을 정확히 알려줘.,reject
98,99,환각 방지,서울대학교 AI대학원과 KAIST AI대학을 교수진 기준으로 비교해줘.,reject
99,100,환각 방지,AI대학 졸업 후 대기업 취업 가능성을 퍼센트로 알려줘.,reject



단일 질문 테스트 시작
질문: 인공지능반도체대학원의 교육 목표를 설명해줘.

[답변]
현재 수집된 자료에는 '인공지능반도체대학원'에 대해 답변할 만큼 충분한 정보가 없습니다.

이 챗봇은 수집된 KAIST AI 관련 학과 자료를 중심으로 답변합니다. 정확하고 최신 정보는 KAIST 공식 홈페이지나 입학처에서 확인하거나, 학과명을 포함해 직접 검색해 주세요.

- KAIST 공식 홈페이지: https://www.kaist.ac.kr/kr/
- KAIST 입학처: https://admission.kaist.ac.kr/home

[출처]
출처 추출 결과 없음

[raw_response 타입]
<class 'dict'>

[raw_response keys]
['question', 'answer', 'analysis', 'route', 'intent', 'department_code', 'needs_clarification', 'sources', 'warnings', 'built_context', 'vector_result', 'sql_result']

100개 질문 평가 시작


  0%|          | 0/100 [00:00<?, ?it/s]

...


카테고리별 평가 요약


,category,auto_eval,count
0,AI대학 소개,PASS_ANSWER,10
1,교수진/연구,PASS_ANSWER,10
2,교육과정,PASS_ANSWER,10
3,입학 정보,PASS_ANSWER,10
4,졸업/이수,PASS_ANSWER,10
5,추천,PASS_ANSWER,10
6,출처 확인,PASS_ANSWER,10
7,학과 비교,PASS_ANSWER,10
8,행정정보,PASS_ANSWER,10
9,환각 방지,FAIL_POSSIBLE_HALLUCINATION,1



기본 지표
전체 질문 수: 100
평균 응답 시간: 3.718 초
최대 응답 시간: 33.831 초
에러 개수: 0
빈 답변 개수: 0
환각 의심 개수: 1

문제 의심 답변 개수: 1


,id,category,question,expected,answer,sources,auto_eval,error
98,99,환각 방지,서울대학교 AI대학원과 KAIST AI대학을 교수진 기준으로 비교해줘.,reject,현재 수집된 자료에는 '서울대학교 AI대학원'에 대해 답변할 만큼 충분한 정보가 없...,,FAIL_POSSIBLE_HALLUCINATION,



예시 답변 확인: 1번, 91번
[ID] 1
[CATEGORY] AI대학 소개
[EXPECTED] answer
[AUTO_EVAL] PASS_ANSWER
[LATENCY] 0.001 sec
----------------------------------------------------------------------------------------------------
[QUESTION]
KAIST AI대학은 어떤 곳이야?
----------------------------------------------------------------------------------------------------
[ANSWER]
제공된 자료에서 질문에 대한 충분한 근거를 찾을 수 없습니다. 학과명이나 알고 싶은 정보 유형을 더 구체적으로 입력해주세요.

정확하고 최신 정보는 KAIST 공식 홈페이지 또는 입학처에서 확인하는 것을 권장합니다.
- KAIST 공식 홈페이지: https://www.kaist.ac.kr/kr/
- KAIST 입학처: https://admission.kaist.ac.kr/home
----------------------------------------------------------------------------------------------------
[SOURCES]

----------------------------------------------------------------------------------------------------
[ERROR]

[ID] 91
[CATEGORY] 환각 방지
[EXPECTED] reject
[AUTO_EVAL] PASS_REJECT
[LATENCY] 0.0 sec
----------------------------------------------------------------------------------------------------
[QUESTION]
AI대학의 올해 경쟁률을 알려줘.

In [2]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("rag_pipeline exists:", (PROJECT_ROOT / "src" / "rag" / "rag_pipeline.py").exists())

CURRENT_DIR: C:\Users\Playdata\workspace\SKN28-third-2TEAM\notebooks
PROJECT_ROOT: C:\Users\Playdata\workspace\SKN28-third-2TEAM
src exists: True
rag_pipeline exists: True


In [3]:
from src.rag.rag_pipeline import create_default_pipeline

pipeline = create_default_pipeline(
    include_sql=True,
    include_debug_context=True,
    preload_vector_retriever=True,
)

q = "AI컴퓨팅학과는 어떤 학과야?"

result = pipeline.run_dict(q, include_debug_context=True)

print("route:", result.get("route"))
print("intent:", result.get("intent"))
print("department_code:", result.get("department_code"))
print("source_count:", len(result.get("sources", [])))
print("warnings:", result.get("warnings"))
print("answer:", result.get("answer")[:500])

print("\nvector_result:")
print(result.get("vector_result"))

route: vector
intent: department_overview
department_code: aic
source_count: 5
warnings: []
answer: AI컴퓨팅학과에 대해 제공된 자료를 종합하면 다음과 같습니다.

핵심 요약:
AI컴퓨팅학과는 인공지능과 컴퓨팅 기술을 중심으로 한 학과로, 학부 및 대학원 과정에서 인공지능 개론, 인간-컴퓨터 상호작용, 리소스 효율적 AI 연구 등 다양한 과목을 제공합니다. 학과설명회도 정기적으로 개최되어 신입생 및 관심자에게 학과 정보를 안내합니다.

교육/연구 방향:
- 인공지능 개론(학부 필수과목)부터 시작하여, 인간-컴퓨터 상호작용 개론(학부 선택과목), 대학원 수준의 인간과 컴퓨터 상호작용, 리소스 효율적 AI 연구 특강 등 심화된 AI 및 컴퓨팅 관련 교육을 진행합니다.
- 리소스 효율적 AI 연구와 같은 최신 AI 기술 및 연구 주제를 다루는 특강을 통해 실무 및 연구 역량을 강화하는 데 중점을 둡니다.

특징:
- 학부와 대학원 과정 모두에서 AI 및 컴퓨팅 관련 과목을 체계적으로 운영합니다.
- 학과설명회를 통해 입학 예정자 및 관심자에게 학과의 교육 내용과 연구 방향을 소개하며, 

vector_result:
{'status': 'searched', 'message': 'Vectorstore 검색이 완료되었습니다.', 'used_query': 'AI컴퓨팅학과는 어떤 학과야? 학과 소개 개요 특징 비전 목표 교육 연구', 'used_filter': {'dept': {'$eq': 'aic'}}, 'used_fallback': False, 'warnings': [], 'search_attempts': [{'search_stage': 'strict_filter', 'metadata_filter': {'dept': {'$eq': 'aic'}}, 'result_count': 25}], 'analysis': {'original_question': 'AI컴퓨팅학과는 어떤 학과야?', 'normal